# 02 · Where the delay minutes come from

**RQ1.** How are reported delay minutes distributed across the five BTS causes, and
how stable is that composition across the panel?

The full-panel carrier + late-aircraft group accounts for 73.3% of reported cause minutes
(95% bootstrap interval 73.0-73.6%); NAS + weather + security account for 26.7%.
These descriptive cause groups do not establish airline control or causal responsibility.
Late aircraft may propagate disruption of any origin. The final method audit will
document the bootstrap resampling unit.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import attribution, viz

raw = io.load_flights()
panel = clean.completed(raw)
universe = clean.attribution_universe(raw)

totals = attribution.cause_totals(universe)
totals[["cause", "family", "minutes", "share"]]

## 2.1 The headline split, with an interval

The resampling unit is the **operating day**, not the flight: flights delayed by one
storm are not independent, so a flight-level bootstrap would produce an interval far
too narrow.

In [ ]:
split = attribution.family_split(universe)
interval = attribution.bootstrap_family_share(universe)

print(f"internal share  {interval['share']:.3f}  "
      f"[{interval['ci_low']:.3f}, {interval['ci_high']:.3f}]  "
      f"over {interval['days']} operating days")
print(f"late aircraft   {split['late_aircraft_share']:.1%} of attributed minutes")

## 2.2 Is the composition stable?

Ten years of a small move is the interesting question: if the interval for one year
clears the interval for another, that is a structural break rather than noise.

In [ ]:
attribution.stability_by_year(universe, bootstrap=True, reps=200)

## 2.3 The figure

A unit waffle, because the question is part-to-whole and the cells can be labelled
individually. Area is a weak channel, so the shares are written in the legend rather
than read off the shape.

In [ ]:
viz.fig_cause_waffle(totals)